# Introduction to N-Gram Language Models

The second half of homework 1 will be different from previous homeworks in this class. You will **not** be interacting with TextWorld or TextWorld-Express for this homework.

Your task for this homework is to create a simple language model based off of unigrams, bigrams, and trigrams. This language model can then be used to generate text that is similar to the text the model is trained on. We will test your solution on hidden test data that will be used to evaluate the accuracy of your implementation. You will be implementing code to build and generate text from all three the models (Unigrams, Bigrams, Trigrams)

For this part of the homework, you will be using NumPy extensively, so please familiarize yourself with the NumPy API (see https://numpy.org/doc/stable/reference/index.html). You are **only** allowed to use a restricted set of libraries for this homework. All packages that come with the default Python installation are permitted, as well as any imports we have already provided for you. You may not use any other libraries than the ones we have provided. If you attempt to use other libraries, the autograder will not be able to run your code.

# Installs and Imports

In [1]:
%pip install numpy

Note: you may need to restart the kernel to use updated packages.


## Imports

In [2]:
# export - DO NOT MODIFY THIS CELL
import numpy as np
import random
from helpers import (
    EOS,
    EOS_token,
    SOS,
    SOS_token,
    Vocab,
    normalize_string,
    make_vocab,
    insert_eos_sos,
)

In [3]:
# export - DO NOT MODIFY OR MOVE THIS LINE
# add any additional imports here (from the Python Standard Library only!)


In [4]:
# DO NOT MODIFY THIS CELL
np.random.seed(42)
random.seed(42)

# Language Models Setup

## Vocabularies for Language Models

A *language model* models the probability of a word to the appearance of prior tokens.

A bigram language model relates the probability of a word to the most recent previous word (for a total of two words modeled): $p(w_n | w_{n-1})$.

A trigram language model relates the probability of a word to the most recent two previous words (for a total of three words modeled): $p(w_n | w_{n-1}, w_{n-2})$.

We will construct a probability matrix for a bigram language model. The probability table will be a 2D table where dimension 0 is the word in position $n-1$ and dimension 1 is the word in position $n$ and dimension 2 is the probability that $w_n$ occurs directly after $w_{n-1}$. Thus, ideally, if our probability matrix is `p` then `p['you', 'are'] = 0.0089`.

But where do we know where each word should go in the matrix? We will assign an unique number to each word, from 0 to $|W|$, the total number of words in our corpus. This number will be referred to as a "token", but it is also the "index" in an arbitrary list of words. Thus is the token for "you" is 2 and the token for "are" is 3, then `p[2, 3] = 0.0089`.

The following creates a "vocabulary" object, which maps the words in a corpus to tokens and vice versa. The vocabulary will have two special tokens, `SOS_token` and `EOS_token`, to indicate the start of a sequence and end of a sequence, respectively. These tokens can be used to demarkate the beginnings and ends of things, which will be useful for generation later.

## Make a Vocabulary

In [5]:
with open("observations.txt", "r") as f:
    observations = f.read()

observations = normalize_string(insert_eos_sos(observations))
print(observations)

SOS you are in the kitchen  EOS SOS in one part of the room you see a stove  EOS SOS there is also an oven  EOS SOS you also see a fridge that is closed  EOS SOS in another part of the room you see a counter that has nothing on it  EOS SOS in one part of the room you see a kitchen cupboard that is closed  EOS SOS there is also a cutlery drawer that is closed  EOS SOS you also see a trash can that is closed  EOS SOS in another part of the room you see a dishwasher that is closed  EOS SOS in one part of the room you see a dining chair that has nothing on it  EOS SOS to the north you see a closed wood door  EOS SOS to the south you see a closed plain door  EOS SOS to the east you see the corridor  EOS SOS inventory maximum capacity is 2 items    your inventory is currently empty  EOS SOS you are in the kitchen  EOS SOS in one part of the room you see a stove  EOS SOS there is also an oven  EOS SOS you also see a fridge that is closed  EOS SOS in another part of the room you see a counter 

In [6]:
VOCAB = make_vocab([observations], "observations")
print("vocab size:", VOCAB.num_words())

vocab size: 108


Map a word to a token

In [7]:
VOCAB.word2token("kitchen")

6

Map a token to a word

In [8]:
VOCAB.token2word(42)

'east'

# Probabilistic Language Modeling **(TO-DO)**

## **Hints:**

*   Use `Vocab.word2token` to get a unique token for each word (or SOS_token/EOS_token)
*   Use `Vocab.token2word` to get the word associated with that paritcular token value
*   Do **not** use `Vocab.word2count`!
*   N-grams should represent the probability P(word_n | word_n-1, word_n-2, ...) for all combinations of words/tokens, so make sure these probabilities make sense (normalize!)

## NumPy

Because we will be using numpy matrices for this homework, you may want to check out the [numpy package documentation](https://numpy.org/doc/stable/). API functions that will be particularly useful are:
- Creating an empty matrix: `np.zeros(shape)` will create a matrix of shape `shape=(size_of_dim_0, size_of_dim_1, ...)` willed entirely with zeros. For example `np.zeros((2, 3))` will create a $2\times 3$ matrix of zeros.
- Getting the size of a matrix: `some_matrix.shape` is the shape of a numpy matrix as a tuple where each element is the size of each dimension.
- Getting an element at a given position can be retrieved using `some_matrix[index1, index2, ...]` or `some_matrix[index1][index2]...`.
- Setting: `some_matrix[index1, index2] = some_value` will set an individual element in a matrix at the given indices. This can also be also be done as `some_matrix[index1][index2]... = some_value`.
- Slicing: sub-matrices can be extracted by giving spans of indices, similar to how python lists work, but with multiple dimensions. For example `some_matrix[5:10,5:10]` will construct a $5\times 5$ square matrix that has elements from the original matrix in positions between $(5,5)$ through $(9,9)$
- Summing: `some_matrix.sum()` adds all the elements of a matrix together. Numpy matrices have a number of built-in mathematical functions like this.
- Mathematical operations: all the normal mathematica operations work as expected in linear algebra. `+` adds element-wise, `*` multiplies element-wise, etc. Matrix shapes must match.
- Type casting: `some_matrix.astype(type)` converts every element in the matrix to a particular type, e.g. `some_matrix.astype(int)`.

## Unigram Language Model **(TO-DO)**

Complete the `make_unigram_model()` function. It takes a text `text` which is a string of text data (normalized to be easy to work with). It also takes in a `Vocab` object. The function should return a 1D NumPy matrix with the probability of seeing a word as the value based on the word's token index from the vocab.

**Notes and Tips:**
* The `text` argument is a single normalized string of sentences with SOS/EOS tokens embedded in it.
* The input text is a single normalized string with SOS/EOS already embedded in the string. You can see how we process it in the provided helper functions. You do not need to do any additional processing on the input text.
* You should be normalizing over the last axis. For bigrams, you should normalize by the sum of all the bigrams that share the same first word w_{n-1}. For trigrams, you should normalize by the sum of all the trigrams that share the same bigram (w_{n-2}, w_{n-1}). For bigrams, normalize over all possible next words given the previous word. For trigrams, normalize over all possible next words given the previous two words. Do not normalize over the entire probability matrix.
* When evaluating on the test data, you may encounter tokens that weren't in the training data, resulting in zero probabilities. The perplexity functions handle this case by using a default value of -10 when encountering zeros.

In [9]:
# export - DO NOT MODIFY OR MOVE THIS LINE
def make_unigram_model(text: str, vocab: Vocab) -> np.ndarray:
    probabilities = np.zeros([vocab.num_words()])
    ### YOUR CODE BELOW HERE

    probabilities = np.zeros([vocab.num_words()])

    words = text.split()

    for word in words:
        token = vocab.word2token(word)
        probabilities[token] += 1

    total_tokens = probabilities.sum()
    if total_tokens > 0:
        probabilities /= total_tokens

    ### YOUR CODE ABOVE HERE
    return probabilities

Make the unigram model

In [10]:
with open("observations.txt", "r") as f:
    observations = f.read()

observations = normalize_string(insert_eos_sos(observations))
VOCAB = make_vocab([observations], "observations")
unigram_lm = make_unigram_model(observations, VOCAB)

The following should be a high-probability word (note: this doesn't mean that the raw probability will be high, but that it will be higher than other, lower-probability words)

In [11]:
unigram_lm[VOCAB.word2token("kitchen")]

np.float64(0.006375595190057299)

### Test Unigram Model

In [12]:
from tests import test_unigram_probabilities

test_unigram_probabilities(build_unigram_function=make_unigram_model)

Beginning sonnet tests
student: 0.02582322357019064 expected: 0.02582322357019064
 - Passed check:  EOS
student: 0.025649913344887348 expected: 0.025649913344887348
 - Passed check:  SOS
student: 0.01750433275563258 expected: 0.01750433275563258
 - Passed check:  thou
student: 0.003119584055459272 expected: 0.003119584055459272
 - Passed check:  beauty
Passed sonnet tests!
n oven  EOS SOS you also see a fridge that is closed  EOS SOS in another part of the room you see a counter that has nothing on it  EOS SOS in one part of the room you see a kitchen cupboard that is closed  EOS SOS there is also a cutlery drawer that is closed  EOS SOS you also see a trash can that is closed  EOS SOS in another part of the room you see a dishwasher that is closed  EOS SOS in one part of the room you see a dining chair that has nothing on it  EOS SOS to the north you see a closed wood door  EOS SOS to the south you see a closed plain door  EOS SOS to the east you see the corridor  EOS SOS inventory ma

## Bigram Language Model **(TO-DO)**

Complete the `make_bigram_model()` function. It takes a text `text` which is a string of text data (normalized to be easy to work with). It also takes in a `Vocab` object. The function should return a 2D numpy matrix where each element is the probability of token of the corresponding index in dimension 0 is followed by the token of the corresponding index in dimension 1.

**Notes and Tips:**
* The `text` argument is a single normalized string of sentences with SOS/EOS tokens embedded in it.
* The input text is a single normalized string with SOS/EOS already embedded in the string. You can see how we process it in the provided helper functions. You do not need to do any additional processing on the input text.
* You should be normalizing over the last axis. For bigrams, you should normalize by the sum of all the bigrams that share the same first word w_{n-1}. For trigrams, you should normalize by the sum of all the trigrams that share the same bigram (w_{n-2}, w_{n-1}). For bigrams, normalize over all possible next words given the previous word. For trigrams, normalize over all possible next words given the previous two words. Do not normalize over the entire probability matrix.
* When evaluating on the test data, you may encounter tokens that weren't in the training data, resulting in zero probabilities. The perplexity functions handle this case by using a default value of -10 when encountering zeros.

In [13]:
# export - DO NOT MODIFY OR MOVE THIS LINE
def make_bigram_model(text: str, vocab: Vocab) -> np.ndarray:
    probabilities = np.zeros([vocab.num_words()] * 2)
    ### YOUR CODE BELOW HERE

    num_words = vocab.num_words()
    probabilities = np.zeros((num_words, num_words))

    words = text.split()

    for i in range(len(words) - 1):
        prev_token = vocab.word2token(words[i])
        next_token = vocab.word2token(words[i+1])
        probabilities[prev_token, next_token] += 1

    row_sum = probabilities.sum(axis=1, keepdims=True)

    row_sum[row_sum == 0] = 1
    probabilities /= row_sum

    ### YOUR CODE ABOVE HERE
    return probabilities

Make the bigram model

In [14]:
with open("observations.txt", "r") as f:
    observations = f.read()

observations = normalize_string(insert_eos_sos(observations))
VOCAB = make_vocab([observations], "observations")
bigram_lm = make_bigram_model(observations, VOCAB)

The following should be a high-probability transition (note: this doesn't mean that the raw probability will be high, but that it will be higher than other, lower-probability transitions)

In [15]:
bigram_lm[VOCAB.word2token("you"), VOCAB.word2token("see")]

np.float64(0.7288359788359788)

The following should be a lower-probability transition.

In [16]:
bigram_lm[VOCAB.word2token("a"), VOCAB.word2token("coin")]

np.float64(0.009419152276295133)

### Test Bigram Model

In [17]:
from tests import test_bigram_probabilities

test_bigram_probabilities(build_bigram_function=make_bigram_model)

Beginning sonnet tests
student: 0.007042253521126761 expected: 0.007042253521126761
 - Passed check:  SOS betwixt
student: 0.07042253521126761 expected: 0.07042253521126761
 - Passed check:  SOS but
student: 0.17391304347826086 expected: 0.17391304347826086
 - Passed check:  me EOS
student: 0.2 expected: 0.2
 - Passed check:  friend EOS
student: 0.17894736842105263 expected: 0.17894736842105263
 - Passed check:  thou art
student: 0.008695652173913044 expected: 0.008695652173913044
 - Passed check:  my joy
Passed sonnet tests!
Beginning observations tests
student: 0.15789473684210525 expected: 0.15789473684210525
 - Passed check:  SOS there
student: 0.2631578947368421 expected: 0.2631578947368421
 - Passed check:  SOS you
student: 0.75 expected: 0.75
 - Passed check:  closed EOS
student: 0.0 expected: 0.0
 - Passed check:  coin EOS
student: 0.6666666666666666 expected: 0.6666666666666666
 - Passed check:  you see
student: 0.15384615384615385 expected: 0.15384615384615385
 - Passed check

## Trigram Language Model **(TO-DO)**

Complete the `make_trigram_model()` function. It takes a text `text` which is a string of sentences. It also takes in a `Vocab` object. The function should return a 3D numpy matrix where each element is the probability of token of the corresponding index in dimension 0 is followed by the token of the corresponding index in dimension 1 then the token of the corresponding index in dimension 2.

**Notes and Tips:**
* The `text` argument is a single normalized string of sentences with SOS/EOS tokens embedded in it.
* The input text is a single normalized string with SOS/EOS already embedded in the string. You can see how we process it in the provided helper functions. You do not need to do any additional processing on the input text.
* You should be normalizing over the last axis. For bigrams, you should normalize by the sum of all the bigrams that share the same first word w_{n-1}. For trigrams, you should normalize by the sum of all the trigrams that share the same bigram (w_{n-2}, w_{n-1}). For bigrams, normalize over all possible next words given the previous word. For trigrams, normalize over all possible next words given the previous two words. Do not normalize over the entire probability matrix.
* When evaluating on the test data, you may encounter tokens that weren't in the training data, resulting in zero probabilities. The perplexity functions handle this case by using a default value of -10 when encountering zeros.
* Conceptually, the 3 dimensions of the trigram matrix represent the three words in a trigram. The 2 dimensions of a bigram matrix represent the two words in a bigram. For generating the second token, look into np.sum() and think about how you can convert the 3d matrix to a 2d matrix to predict the second token based on the first token similar to the Bigram model generation.  From the third token onwards, use the 2 previous tokens to predict with the Trigram table.
* The 3D matrix for all the trigrams contains probabilities for every combination of the trigrams. There are probably a lot of trigrams like ("in", "you", "are") that don't make sense and would have a count of 0. It is a sparse matrix and you cannot see all the rows currently from just printing out the matrix. To see the unique probabilities values, try doing something like `numpy.unique(trigram_lm.flatten())`. This will collapse all the 3D values into 1D, and only give unique values. This will tell you all the probability values. To be more thorough, you can loop through all indices of the matrix which are nonzero, and print their corresponding trigram.


In [123]:
# export - DO NOT MODIFY OR MOVE THIS LINE
def make_trigram_model(text: str, vocab: Vocab) -> np.ndarray:
    probabilities = np.zeros([vocab.num_words()] * 3)
    ### YOUR CODE BELOW HERE
    tokens = [vocab.word2token(w) for w in text.split()]

    for i in range(len(tokens) - 2):
        w1, w2, w3 = tokens[i], tokens[i + 1], tokens[i + 2]
        probabilities[w1, w2, w3] += 1

    for w1 in range(vocab.num_words()):
        for w2 in range(vocab.num_words()):
            total = np.sum(probabilities[w1, w2, :])
            if total > 0:
                probabilities[w1, w2, :] /= total

    ### YOUR CODE ABOVE HERE
    return probabilities

In [124]:
with open("observations.txt", "r") as f:
    observations = f.read()

observations = normalize_string(insert_eos_sos(observations))
VOCAB = make_vocab([observations], "observations")
trigram_lm = make_trigram_model(observations, VOCAB)

In [125]:
trigram_lm[VOCAB.word2token("you"), VOCAB.word2token("see"), VOCAB.word2token("a")]

np.float64(0.7368421052631579)

### Test Trigam Model

In [126]:
from tests import test_trigram_probabilities

test_trigram_probabilities(build_trigram_function=make_trigram_model)

Beginning sonnet tests
student: 1.0 expected: 1.0
 - Passed check:  SOS thou art
student: 0.16666666666666666 expected: 0.16666666666666666
 - Passed check:  SOS how heavy
student: 1.0 expected: 1.0
 - Passed check:  so dear EOS
student: 0.5 expected: 0.5
 - Passed check:  of me EOS
student: 0.3333333333333333 expected: 0.3333333333333333
 - Passed check:  thou wilt be
student: 0.14285714285714285 expected: 0.14285714285714285
 - Passed check:  my love thou
Passed sonnet tests!
Beginning observations tests
student: 1.0 expected: 1.0
 - Passed check:  SOS there is
student: 0.5714285714285714 expected: 0.5714285714285714
 - Passed check:  SOS in one
student: 1.0 expected: 1.0
 - Passed check:  wood door EOS
student: 1.0 expected: 1.0
 - Passed check:  is closed EOS
student: 0.9 expected: 0.9
 - Passed check:  you see a
student: 0.16666666666666666 expected: 0.16666666666666666
 - Passed check:  see a stove
Passed observations tests!
All tests passed!


# Text Generation **(TO-DO)**

### Unigram Generation

Complete the following function to generate text from a unigram language model.
`generate_from_unigram()` will take the following parameters:
- `first_token`: the token that will appear first in the generated text (e.g., `SOS_token`).
- `probabilities`: a unigram model, a numpy 1D array of probability values for each token.
- `vocab`: a Vocab object.
- `max_length`: the maximum number of words to generate.

The function should generate words until either `max_length` words is reached or until an `EOS_token` is generated. Your generated text should always end with an `EOS_token`. DO NOT hard code any strings. Use the helper constants provided.

The return value will be a list of words where the first word is always the first word that corresponds to `first_token`.

**Notes:**
* You will want to sample the probability distributions that you generated from earlier functions when creating the unigram, bigram, and trigram models. You might find `nump` [random sampling functions](https://numpy.org/doc/stable/reference/random/generated/numpy.random.choice.html) helpful. 
* You should never hardcode the strings ‘EOS’ or ‘SOS’. You should always use the helper constants when you want to reference the end/start of sequence tokens or strings.

In [156]:
# export - DO NOT MODIFY OR MOVE THIS LINE
def generate_from_unigram(
    first_token: int, probabilities: np.ndarray, vocab: Vocab, max_length: int
) -> list:
    # An empty list of words to populate
    words = [vocab.token2word(first_token)]
    ### YOUR CODE BELOW HERE
    probs = probabilities.copy().astype(float)
    probs[SOS_token] = 0.0
    probs = probs / probs.sum()
    
    for _ in range(max_length - 1):
        next_token = np.random.choice(len(probs), p=probs)
        words.append(vocab.token2word(next_token))

        if next_token == EOS_token:
            break
    

    if words[-1] != vocab.token2word(EOS_token):
        words[-1] = vocab.token2word(EOS_token)

    ### YOUR CODE ABOVE HERE
    return words

In [157]:
" ".join(generate_from_unigram(SOS_token, unigram_lm, VOCAB, max_length=128))

'SOS EOS'

In [158]:
from tests import test_generate_from_unigram

test_generate_from_unigram(
    generate_from_unigram_function=generate_from_unigram,
    build_unigram_function=make_unigram_model,
)

['SOS', 'middle', 'dies', 'golden', 'look', 'to', 'love', 'make', 'this', 'others', 'death', 'of', 'usury', 'the', 'lose', 'removed', 'though', 'boast', 'in', 'lives', 'and', 'but', 'in', 'a', 'shalt', 'eyes', 'on', 'thine', 'that', 'which', 'to', 'i', 'walls', 'lives', 'for', 'wide', 'pleasure', 'all', 'themselves', 'suffering', 'their', 'wealth', 'time', 'are', 'but', 'with', 'scornd', 'leaving', 'entertain', 'EOS']
['SOS', 'with', 'EOS']


AssertionError: 

### Bigram Generation

Complete the following function to generate text from a unigram language model.
`generate_from_bigram()` will take the following parameters:
- `first_token`: the token that will appear first in the generated text (e.g., `SOS_token`).
- `probabilities`: a bigram model, a numpy 2D array of probability values where `probabilities[i][j]` indicates the probability of token `i` followed by token `j`.
- `vocab`: a Vocab object.
- `max_length`: the maximum number of words to generate.

The function should generate words until either `max_length` words is reached or until an `EOS_token` is generated. Your generated text should always end with an `EOS_token`. DO NOT hard code any strings. Use the helper constants provided.

The return value will be a list of words where the first word is always the first word that corresponds to `first_token`.

To implement this function, start by sampling a word based on the probability of occurring after `first_token`. Then continue to sample tokens based on their probability of occurring after each subsequently generated token. Terminate when `EOS_token` is generated or when the maximum length of tokens is generated. Convert each token into a word.

You might find the [Numpy random sampling functions](https://numpy.org/doc/stable/reference/random/index.html) helpful.

In [97]:
# export - DO NOT MODIFY OR MOVE THIS LINE
def generate_from_bigram(
    first_token: int, probabilities: np.ndarray, vocab: Vocab, max_length: int
) -> list:
    # An empty list of words to populate
    words = [vocab.token2word(first_token)]
    ### YOUR CODE BELOW HERE
    words_idx = np.arange(vocab.num_words())
    word_id = first_token
    for _ in range(max_length - 1):
        word_id = np.random.choice(words_idx, p=probabilities[word_id])
        w = vocab.token2word(word_id)
        words.append(w)
        if w == vocab.token2word(EOS_token):
            break
        
    if words[-1] != vocab.token2word(EOS_token):
        words[-1] = vocab.token2word(EOS_token)
    ### YOUR CODE ABOVE HERE
    return words

Generate from the bigram model.

In [98]:
" ".join(generate_from_bigram(SOS_token, bigram_lm, VOCAB, max_length=128))

'SOS in the room you see a umbrella stand that is closed EOS'

In [99]:
from tests import test_generate_from_bigram

test_generate_from_bigram(
    generate_from_bigram_function=generate_from_bigram,
    build_bigram_function=make_bigram_model,
)

['SOS', 'how', 'would', 'bar', 'my', 'way', 'so', 'EOS']
['SOS', 'but', 'EOS']
All tests passed!


### Trigram Generation

Complete the following function to generate text from a unigram language model.
`generate_from_trigram()` will take the following parameters:
- `first_token`: the token that will appear first in the generated text (e.g., `SOS_token`).
- `probabilities`: a trigram model, a numpy 3D array of probability values where `probabilities[i][j][k]` indicates the probability of token `i` followed by token `j` followed by token `k`.
- `vocab`: a Vocab object.
- `max_length`: the maximum number of words to generate.

The function should generate words until either `max_length` words is reached or until an `EOS_token` is generated. Your generated text should always end with an `EOS_token`. DO NOT hard code any strings. Use the helper constants provided.

The return value will be a list of words where the first word is always the first word that corresponds to `first_token`.

To implement this function, start by sampling a word based on the probability of occurring after `first_token`. To do this you will need to figure out how to make the trigram model operate like a bigram model. Once you have a sequence of two tokens, you can proceed by sampling a next token based on its probability of occurring after the token at position $n-2$ and token at position $n-1$. If the probability vector for the current sequence of two tokens is 0, you should select the next token randomly. Terminate when `EOS_token` is generated or when the maximum length of tokens is generated. Convert each token into a word.

You might find the [Numpy random sampling functions](https://numpy.org/doc/stable/reference/random/index.html) helpful.

In [ ]:
# export - DO NOT MODIFY OR MOVE THIS LINE
def generate_from_trigram(
    first_token: int, probabilities: np.ndarray, vocab: Vocab, max_length: int
) -> list:
    # An empty list of words to populate
    words = []
    ### YOUR CODE BELOW HERE ###
    count = 0
    word_ids = np.arange(vocab.num_words())

    word = vocab.token2word(first_token)
    prev_word_id = SOS_token
    word_id = first_token

    def normalize_probs(probs: np.ndarray) -> np.ndarray:
        norm_probs = np.zeros_like(probs, dtype=float)
        V = probs.shape[2]
        for i in range(probs.shape[0]):
            for j in range(probs.shape[1]):
                total = np.sum(probs[i, j])
                if total > 0:
                    norm_probs[i, j] = probs[i, j] / total
                else:
                    norm_probs[i, j] = np.ones(V) / V
        return norm_probs

    norm_probs = normalize_probs(probabilities)

    while count < max_length and word != vocab.token2word(EOS_token):
        words.append(word)
        next_word_id = np.random.choice(word_ids, p=norm_probs[prev_word_id, word_id])

        word = vocab.token2word(next_word_id)
        prev_word_id = word_id
        word_id = next_word_id
        count += 1

    if len(words) == 0:
        words.append(vocab.token2word(EOS_token))
    elif words[-1] != vocab.token2word(EOS_token):
        words[-1] = vocab.token2word(EOS_token)

    ### YOUR CODE ABOVE HERE ###
    return words

Generate from the trigram model.

In [171]:
" ".join(generate_from_trigram(SOS_token, trigram_lm, VOCAB, max_length=256))

'SOS laundry folding coin it wardrobe street EOS'

In [175]:
from tests import test_generate_from_trigram

test_generate_from_trigram(
    generate_from_trigram_function=generate_from_trigram,
    build_trigram_function=make_trigram_model,
)

MemoryError: Unable to allocate 26.9 GiB for an array with shape (1534, 1534, 1534) and data type float64

## Perplexity

*Perplexity* is a measure of how well a language model captures a dataset. It is a measure of how "surprised" the model is when it sees a sequence. If a model has perplexity close to zero, it means that any sequence you throw at the model is very probable according to the model. If the perplexity is high, it means that the sequences it is seeing seem very improbable according to the model. If the sequences are real data from the corpus, then a low perplexity is good because the model is not being surprised by real data. Thus, lower is better.

The formula for trigram perplexity is $ppl(w)=exp\big(-\frac{1}{n}\displaystyle\sum\limits_{t=1}^{n}\log p(w_t|w_{t-1}, w_{t-2})\big)$ where $w$ is a word vector conisting of $n$ words $w_1...w_n$. The $log$ maps values between $0...1$ to a scale from $-∞...0$ to avoid multiplying small probability values together. We must add log probabilities instead of multiplying regular probabilities. The $\frac{1}{n}$ gives us the average log probability. The $-1$ converts log probabilities from numbers less than 0 to numbers greater than 0 so that unlikely word vectors (probabilities close to 0 or log probabilities close to $-∞$) become very high positive numbers (high perplexity). This is also called the *negative log likelihood*, which, counter-intuitively is a positive score.

To get the final perplexity score, you would then apply a $exp(⋅)$ to map the values in the log scale back to non-log scale.

To summarize: convert probabilities to log-scale, combine probability chains, get the mean, flip to positive valued numbers, then covert to non-log-scale.

We will be making one slight modification to this: we will **not** be converting back to non-log scale. This is because perplexity is mainly used to compare different text generation schemes to each other. Since log perplexity can be compared in the same way perplexity can, we will not be using an $exp(⋅)$ in our calculations.

In [ ]:
from helpers import perplexity1, perplexity2, perplexity3

with open("observations.txt", "r") as f:
    observations = f.read()
observations = normalize_string(insert_eos_sos(observations))
VOCAB = make_vocab([observations], "observations")

seq = [VOCAB.word2token(w) for w in observations.split()]

observation_unigram_lm = make_unigram_model(observations, VOCAB)
observation_bigram_lm = make_bigram_model(observations, VOCAB)
observation_trigram_lm = make_trigram_model(observations, VOCAB)

print(perplexity1(seq, observation_unigram_lm))
print(perplexity2(seq, observation_unigram_lm, observation_bigram_lm))
print(
    perplexity3(
        seq, observation_unigram_lm, observation_bigram_lm, observation_trigram_lm
    )
)

In [ ]:
with open("sonnets.txt", "r") as f:
    sonnets = f.read()
sonnets = normalize_string(insert_eos_sos(sonnets))
VOCAB = make_vocab([sonnets], "sonnets")

seq = [VOCAB.word2token(w) for w in sonnets.split()]

sonnets_unigram_lm = make_unigram_model(sonnets, VOCAB)
sonnets_bigram_lm = make_bigram_model(sonnets, VOCAB)
sonnets_trigram_lm = make_trigram_model(sonnets, VOCAB)

print(perplexity1(seq, sonnets_unigram_lm))
print(perplexity2(seq, sonnets_unigram_lm, sonnets_bigram_lm))
print(perplexity3(seq, sonnets_unigram_lm, sonnets_bigram_lm, sonnets_trigram_lm))

## Write text with perplexity **(TO-DO)**

For this exercise, create two text sequences of your own design that have a perplexity above or below a required value. This exercise uses the observations.txt file for this sequence so all the words must exist in the observations vocabulary. Based on your understanding of perplexity, can you craft a sequence with the desired perplexity? If you get an out of bounds error, try adding a punctuation mark to the end. The text processing helpers insert SOS/EOS around full sentences. You don’t have to write a full sentence, just end it with `.`, `!`, or `?`. 


**Note:** this exercise is purely for exploration purposes. It is not graded. 

In [ ]:
perplexity_under_3 = "REPLACE ME!"
perplexity_over_7 = "REPLACE ME!"

In [ ]:
with open("observations.txt", "r") as f:
    observations = f.read()
observations = normalize_string(insert_eos_sos(observations))
VOCAB = make_vocab([observations], "observations")

observation_unigram_lm = make_unigram_model(observations, VOCAB)
observation_bigram_lm = make_bigram_model(observations, VOCAB)
observation_trigram_lm = make_trigram_model(observations, VOCAB)

In [ ]:
low_perplexity = normalize_string(insert_eos_sos(perplexity_under_3))
low_seq = [VOCAB.word2token(w) for w in low_perplexity.split()]

print(
    perplexity3(
        low_seq, observation_unigram_lm, observation_bigram_lm, observation_trigram_lm
    )
)

In [ ]:
high_perplexity = normalize_string(insert_eos_sos(perplexity_over_7))
high_seq = [VOCAB.word2token(w) for w in high_perplexity.split()]

print(
    perplexity3(
        high_seq, observation_unigram_lm, observation_bigram_lm, observation_trigram_lm
    )
)

### Test your sequences

In [ ]:
from tests import test_perplexity

test_perplexity(
    perplexity_under_3,
    perplexity_over_7,
    build_unigram_function=make_unigram_model,
    build_bigram_function=make_bigram_model,
    build_trigram_function=make_trigram_model,
)

# Testing

## Evaluate student models

We will evaluate your models by training them on half of the data, and testing perplexity on the remaining half. To build your own tests, you can partition the given text files differently or download your own text files. You may share your results for any new text files with other students. As a reminder, you are **not** permitted to share code with other students.

In [ ]:
with open("observations.txt", "r") as f:
    observations = f.read()

sil = normalize_string(insert_eos_sos(observations))
sil = " ".join(sil.split()[:2000])
sil_train = " ".join(sil.split()[:1500])
sil_test = " ".join(sil.split()[1500:])

In [ ]:
sil_vocab = make_vocab([sil], "sil")
sil_vocab.num_words()

Build Student Models on the training data

In [ ]:
unigram_lm = make_unigram_model(sil_train, sil_vocab)
bigram_lm = make_bigram_model(sil_train, sil_vocab)
trigram_lm = make_trigram_model(sil_train, sil_vocab)

Evaluate on Training Data

In [ ]:
print("Training Results")

token_sequence = [sil_vocab.word2token(w) for w in sil_train.split()]

print("Unigram perplexity: ", perplexity1(token_sequence, unigram_lm))
print("Bigram perplexity: ", perplexity2(token_sequence, unigram_lm, bigram_lm))
print(
    "Trigram perplexity: ",
    perplexity3(token_sequence, unigram_lm, bigram_lm, trigram_lm),
)

Evaluate on Testing Data

In [ ]:
print("Testing Results")

testing_token_seq = [sil_vocab.word2token(w) for w in sil_test.split()]

print("Unigram perplexity: ", perplexity1(testing_token_seq, unigram_lm))
print("Bigram perplexity: ", perplexity2(testing_token_seq, unigram_lm, bigram_lm))
print(
    "Trigram perplexity: ",
    perplexity3(testing_token_seq, unigram_lm, bigram_lm, trigram_lm),
)

Run tests to ensure your perplexity falls within the acceptable bounds

In [ ]:
from tests import test_model_perplexity

test_model_perplexity(
    build_unigram_function=make_unigram_model,
    build_bigram_function=make_bigram_model,
    build_trigram_function=make_trigram_model,
)

# Grading

We will be grading your submission based on your probability tables, text generation, as well as the perplexity of your code evaluated on randomly choosen text data (it will be similar to the dataset we provide in this notebook). These tests will be very similar to the tests provided but on a different dataset.

Part B is worth 50 points based on your performance across several trials of randomly sampled text data (from the same data provided), and your overall score will be weighted as follows:
- 10 points for correctly calculating unigram probabilities
- 10 points for correctly calculating bigram probabilities
- 10 points for correctly calculating trigram probabilities
- 10 points for correctly generating text from your models
- 10 points for appropriate perplexity for all three models

Do not try to subvert or game the autograder, as these cases will lead to a score of 0. We have a built-in curve to our autograder to help student scores. As such, the sanity checks provided do not guarantee similar performance on the overall homework, but they do help ensure that your code runs correctly as you intend.

# Submission

Upload this notebook with the name `submission.ipynb` file to Gradescope. The autograder will **only** run successfully if your file is named this way. You must ensure that you have removed all print statements from **your** code, or the autograder may fail to run. Excessive print statements will also result in muddled test case outputs, which makes it more difficult to interpret your score. 

We've added appropriate comments to the top of certain cells for the autograder to export (`# export`). You do NOT have to do anything (e.g. remove print statements) to cells we have provided - anything related to those have been handled for you. You are responsible for ensuring your own code has no syntax errors or unnecessary print statements. You ***CANNOT*** modify the export comments at the top of the cells, or the autograder will fail to run on your submission.

You should ***not*** add any cells that your code requires to the notebook when submitting. You're welcome to add any code as you need to extra cells when testing, but they will not be graded. Only the provided cells will be graded. As mentioned in the top of the notebook, **any helper functions that you add should be nested within the function that uses them.**

If you encounter any issues with the autograder, please feel free to make a post on Ed Discussion. We highly recommend making a public post to clarify any questions, as it's likely that other students have the same questions as you! If you have a question that needs to be private, please make a private post.